# Full NYT10m validation: supplied entities, GLiNER-relex and GLiDRE

This notebook compares **GLiNER-relex base, GLiNER-relex large and GLiDRE large** on the same frozen **36,266 validation bags / 44,751 selected sentence records**. A bag is an ordered head → tail pair. All models receive the exact dataset mentions and the same deterministic cap of 20 records; entity recognition is not part of this experiment.

**Default execution:** base=False, large=False, GLiDRE=True. Run All downloads the GLiDRE assets at the beginning, verifies compatibility offline, runs GLiDRE sequentially, and recomputes comparisons using the saved base/large scores. The original 72,532 relex results remain unchanged. Each model flag is independent. Enabling a relex flag creates a separate rerun; an incomplete rerun is resumable.

**Internet is needed only for section 2.** After `PREFETCH COMPLETE`, subsequent model stages use explicit offline mode. GLiDRE's helper package is isolated in a subprocess; the project environment is not downgraded.

Confirmed annotations are loaded fresh from the two public annotation JSON files. Their hashes are recorded with every report; editing labels does not invalidate inference. Full validation uses noisy distant supervision. The reviewed 30 remain a selected, repeatedly used text diagnostic. The official test is not read.

No GLiDRE results are included yet. Its actual weight loading, parameter count, native parity and longest-input checks execute when this notebook is run.

## 1. Execution flags, kernel and annotation reference

All model and download flags default to False. Default Run All verifies the public confirmed annotations and displays a matching published comparison without detailed caches or model loading. `ANALYZE_LOCAL_SCORES=True` enables local cache analysis; each model has a separate execution flag. Model stages require the project `.venv` kernel. Annotation Markdown edits alone do not change the scored JSON reference.


In [2]:
# Published reports are the default; enable local stages explicitly.
RUN_RELEX_BASE = False
RUN_RELEX_LARGE = False
RUN_GLIDRE = False
FETCH_ENABLED_MODEL_ASSETS = False
ANALYZE_LOCAL_SCORES = False
SAVE_COMPACT_COMPARISON = False

import importlib
import json
import os
import shutil
import signal
import subprocess
import sys
from pathlib import Path
import pandas as pd
from IPython.display import HTML, Markdown, display

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / 'source/validation_model_comparison.py').is_file()
             and (p / 'source/Full_validation_relex.ipynb').is_file()), None)
if ROOT is None:
    raise FileNotFoundError('Launch this notebook from the repository or source directory.')
if str(ROOT / 'source') not in sys.path:
    sys.path.insert(0, str(ROOT / 'source'))
PYTHON = ROOT / '.venv/bin/python'
if any((RUN_RELEX_BASE, RUN_RELEX_LARGE, RUN_GLIDRE)):
    if not PYTHON.is_file() or Path(sys.prefix).resolve() != (ROOT / '.venv').resolve():
        raise RuntimeError('Select the project .venv kernel before running model stages.')

import reviewed_validation_reference as reference_reader
reference_reader = importlib.reload(reference_reader)
confirmed_labels, reference_metadata = reference_reader.load_reviewed_reference(ROOT)
print('Kernel:', sys.executable)
print('Model flags:', {'base': RUN_RELEX_BASE, 'large': RUN_RELEX_LARGE, 'glidre': RUN_GLIDRE})
print('Reference version:', reference_metadata['version'])
print('Reference labels SHA-256:', reference_metadata['labels_sha256'])
print('Confirmed positive instances:', sum(len(v) for v in confirmed_labels.values()))
print('Changed samples relative to the original reviewed 30:',
      reference_metadata['changed_samples_from_original_reviewed30'])
for relative in reference_reader.ANNOTATION_FILES:
    print('Annotation source:', relative)

# Stream a separate worker; old GLiNER helpers never enter the notebook kernel.
def run_stage(stage, *, offline=True, keep_awake=False, extra_args=()):
    command = [str(PYTHON), str(ROOT / 'source/run_full_validation_glidre.py'), stage, *extra_args]
    environment = os.environ.copy()
    environment.update({'PYTHONUNBUFFERED': '1', 'TOKENIZERS_PARALLELISM': 'false',
                        'HF_HUB_DISABLE_TELEMETRY': '1', 'HF_HUB_DISABLE_XET': '1',
                        'HF_HUB_DOWNLOAD_TIMEOUT': '60', 'HF_HUB_ETAG_TIMEOUT': '30'})
    for key in ('HF_HUB_OFFLINE', 'TRANSFORMERS_OFFLINE', 'HF_DATASETS_OFFLINE'):
        if offline:
            environment[key] = '1'
        else:
            environment.pop(key, None)
    if keep_awake and shutil.which('caffeinate'):
        command = ['caffeinate', '-is', *command]
    print('Starting:', stage, '| offline:', offline, flush=True)
    process = subprocess.Popen(command, cwd=ROOT, env=environment, stdout=subprocess.PIPE,
                               stderr=subprocess.STDOUT, text=True, bufsize=1, start_new_session=True)
    try:
        for line in process.stdout:
            print(line, end='')
        return_code = process.wait()
    except KeyboardInterrupt:
        os.killpg(process.pid, signal.SIGINT)
        try:
            process.wait(timeout=30)
        except subprocess.TimeoutExpired:
            os.killpg(process.pid, signal.SIGKILL)
            process.wait()
        raise
    if return_code:
        raise RuntimeError(f'{stage} failed with exit code {return_code}; later stages were not started.')


Kernel: /Users/alessandro/Progetti/Ensembling/.venv/bin/python
Model flags: {'base': False, 'large': False, 'glidre': False}
Reference version: text_supported_inference_v2_20261006
Reference labels SHA-256: fb9972e04f461ce077ccde2c10d7811496fb711614be8d14344dde986fdb46ea
Confirmed positive instances: 24
Changed samples relative to the original reviewed 30: ['N18']
Annotation source: support/validation_clean_annotations_2026-09-29.json
Annotation source: support/validation_expanded_18_annotations_2026-10-01.json


## 2. Initial asset prefetch

Only assets for enabled models are fetched. GLiDRE uses the pinned `cea-list-ia/glidre_large` revision: approximately **3.20 GB** of weights containing both encoders, its text tokenizer, the BGE label tokenizer, pinned GLiDRE source and a GLiNER 0.2.13 helper wheel. BGE backbone weights are not downloaded separately. The helper wheel is extracted into a private import directory; no package installation changes `.venv`.

Assets reside in `data/model_cache/`. A receipt records file checksums, including the Hub weight checksum. With both relex flags False, their retained weights are neither downloaded nor loaded. Disconnection is possible after this cell prints `PREFETCH COMPLETE`.

In [3]:
# Complete all network operations before the offline experiment starts.
enabled_relex = [name for name, enabled in (('base', RUN_RELEX_BASE), ('large', RUN_RELEX_LARGE)) if enabled]
if FETCH_ENABLED_MODEL_ASSETS and (RUN_GLIDRE or enabled_relex):
    arguments = ['--glidre'] if RUN_GLIDRE else []
    if enabled_relex:
        arguments += ['--relex-members', *enabled_relex]
    run_stage('prefetch', offline=False, extra_args=arguments)
else:
    print('Prefetch disabled; enabled inference requires assets already present locally.')


Prefetch disabled; enabled inference requires assets already present locally.


## 3. Offline GLiDRE compatibility checks

Preparation binds GLiDRE to the original validation manifest, model/source revisions and unchanged relation descriptions. Descriptions are uppercased with spaces replaced by underscores for GLiDRE; the output IDs remain the same 24 positive relations. No reviewed or distant answers enter model inputs.

The native GLiDRE processor has a **512-word preprocessing cap**. This experiment raises that guard to 2,048 words and explicitly disables tokenizer truncation, preserving the full selected sentences. This changes no pretrained weights or positional layers. The text encoder uses relative positions; the label encoder's own context limit remains enforced.

The pinned author code contains inconsistent unpacking between its bi-encoder, word-attention extractor and relation head. A local compatibility bridge reconnects their existing outputs, including the attention tensor, without replacing layers or changing weights. This bridge is included in the run fingerprint. Its tensor routing and native preprocessing/decoder parity have been checked on a tiny synthetic architecture with random weights; the released checkpoint checks remain below.

The preflight strictly loads every checkpoint tensor, verifies the complete parameter count is below 1B, checks native directed-score/decoder parity and cached-label parity, aligns all 44,751 selected sentences, and performs a real forward pass on the longest tokenized input. Word boundaries are refined only at supplied mention offsets. A failed check stops the experiment before full inference. A matching successful preflight can be reused.

In [4]:
# No model is loaded unless its individual execution flag is enabled.
if RUN_GLIDRE:
    run_stage('prepare', offline=True)
    run_stage('offline-check', offline=True, keep_awake=True)
else:
    print('GLiDRE preparation and model preflight disabled.')


GLiDRE preparation and model preflight disabled.


## 4. Sequential inference and resumption

Enabled relex members run first, one at a time, and are released before GLiDRE. By default all three model flags are False and no model is called. Optional relex reruns use `data/pilot_runs/relex_explicit_reruns_20261006/`; completed reruns are archived before an explicitly requested repeat. The original relex run stays intact.

GLiDRE stores all 24 sentence-level sigmoid scores for the supplied head → tail pair, bag maxima and timings in `data/pilot_runs/glidre_full_validation_given_entities_v1_20261006/`. Fixed label embeddings are computed once per worker session through the native label encoder. The relation layers and checkpoint weights remain unchanged. Every bag supports multiple positive labels or an empty set.

Completed valid GLiDRE bags are skipped only after fingerprint, input, span, score-inventory and bag-maxima checks. Failed attempts remain accounted for in timings. Three consecutive failures or less than 2 GiB free disk stop execution. On macOS, `caffeinate` prevents idle sleep during enabled model stages.

In [5]:
# Execute only enabled members; GLiDRE's complete saved bags remain resumable.
model_run_overrides = {}
for name, enabled in (('base', RUN_RELEX_BASE), ('large', RUN_RELEX_LARGE)):
    if enabled:
        run_stage('run-' + name, offline=True, keep_awake=True)
        model_run_overrides[name] = ROOT / 'data/pilot_runs/relex_explicit_reruns_20261006' / name
if RUN_GLIDRE:
    run_stage('run', offline=True, keep_awake=True)
else:
    print('GLiDRE model inference disabled.')


GLiDRE model inference disabled.


## 5. Fresh annotation scoring and complete-cache checks

Default mode reads a published comparison matching the current confirmed annotation metadata. Opt-in local analysis loads cached scores and current confirmed JSON annotations independently of the frozen inference. A label edit triggers new reviewed scores without rerunning any model. Reports are saved under the current annotation-label hash; previous changed reports are archived. Missing, failed, mismatched or incomplete model results cannot be reported as a complete experiment.

The fixed grid uses strict **score > cutoff** at 0.5, 0.7 and 0.9: individual models, each pair's union/intersection under all nine member-cutoff combinations, and a label-wise two-of-three vote under all 27 combinations. The full three-model grid has **90 configurations per reference**. Scores are not averaged across differently calibrated models. These are development diagnostics, not automatic selection or independent confirmation of the best configuration.

Ensemble cost sums all member inference times. GLiDRE includes session label encoding and previous failed attempts; checkpoint loading and compatibility checks are displayed separately. Full-validation distant labels and the reviewed 30 are never pooled. Train-seen/unseen cohorts are omitted.

In [6]:
# Recompute analysis only; no model import, download or inference occurs here.
import validation_model_comparison as comparison_module
comparison_module = importlib.reload(comparison_module)
comparison = None
reviewed_bags = []
if ANALYZE_LOCAL_SCORES:
    comparison, reviewed_bags = comparison_module.build_comparison(
        require_glidre=RUN_GLIDRE,
        model_run_overrides=model_run_overrides,
        save=SAVE_COMPACT_COMPARISON,
    )
else:
    # A portable checkout can display only a report matching the current annotations.
    candidates = sorted((ROOT / 'results').glob('validation_relex*_comparison_*.json'),
                        key=lambda p: p.stat().st_mtime, reverse=True)
    candidates += sorted((ROOT / 'results').glob('validation_relex_current_annotations_*.json'),
                         key=lambda p: p.stat().st_mtime, reverse=True)
    for candidate in candidates:
        saved = json.loads(candidate.read_text())
        if saved.get('reviewed_reference_metadata') == reference_metadata:
            comparison = saved
            print('Published comparison:', candidate.relative_to(ROOT))
            break
    if comparison is None:
        print('No published comparison matches the current annotations; local scoring is required.')
if comparison is not None:
    print('Models with complete caches:', comparison['models'])
    print('Configurations per reference:', comparison['configurations_per_reference'])
    print('Annotation label hash:', comparison['reviewed_reference_metadata']['labels_sha256'])
    print('Reference:', comparison['reference_warning'])
    print('Compact report:', comparison.get('published_path', 'loaded published report'))


Published comparison: results/validation_relex_glidre_comparison_fb9972e04f46.json
Models with complete caches: ['base', 'large', 'glidre']
Configurations per reference: 90
Annotation label hash: fb9972e04f461ce077ccde2c10d7811496fb711614be8d14344dde986fdb46ea
Reference: Distant validation labels and selected reused reviewed-30 annotations are separate development references. No automatic model selection or official test evaluation.
Compact report: loaded published report


## 6. Primary results and model costs

At the common 0.5 cutoff, tables show each individual, each pair's union/intersection, and the three-member majority vote when GLiDRE is complete. **TP** counts correct positive relations; **FP** counts extra relations; **FN** counts missed relations. Precision, recall and micro-F1 pool positive label decisions across bags. Exact match requires the complete bag label set to match, including empty sets.

`negative_bags_with_fp` uses the reference shown: distant `NA` is not independently verified absence of a relation. `total_member_inference_s` is the full-validation member cost for that configuration, even in the reviewed-30 diagnostic table.

In [7]:
# Display primary metrics with every available member included in deployment cost.
def show_table(frame, *, max_rows=30):
    with pd.option_context('display.max_rows', max_rows, 'display.max_columns', None,
                          'display.max_colwidth', 90, 'display.float_format', '{:.4f}'.format):
        display(frame)

if comparison is not None:
    grid = pd.DataFrame(comparison['rows'])
    metric_columns = ['configuration', 'bags', 'TP', 'FP', 'FN', 'precision', 'recall',
                      'micro_f1', 'exact_match_bags', 'negative_bags_with_fp', 'total_member_inference_s']
    primary = grid[grid['cutoffs'].map(lambda values: all(c == .5 for c in values))]
    for reference in ('all_validation_distant_labels', 'reviewed_30_text'):
        display(Markdown('**' + reference + '**'))
        show_table(primary[primary['reference'] == reference][metric_columns].reset_index(drop=True))
    timing = comparison['timing']
    show_table(pd.DataFrame([{'model': model,
        'total_inference_s': timing['inference_s_by_model'][model],
        'mean_bag_wall_s': timing['bag_mean_s'][model],
        'p95_bag_wall_s': timing['bag_p95_s'][model]}
        for model in comparison['models']]))
    runtime = comparison.get('glidre_runtime', {})
    if runtime.get('sessions'):
        show_table(pd.DataFrame(runtime['sessions']))
    preflight = comparison.get('glidre_preflight', {})
    if preflight:
        print('GLiDRE compatibility preflight seconds (excluded from inference cost):', preflight['wall_s'])
    print('Timing scope:', timing['scope'])


**all_validation_distant_labels**

,configuration,bags,TP,FP,FN,precision,recall,micro_f1,exact_match_bags,negative_bags_with_fp,total_member_inference_s
0,single|base@0.5,36266,2136,7074,1926,0.2319,0.5258,0.3219,28921,4873,3157.3400
1,single|large@0.5,36266,2698,9343,1364,0.2241,0.6642,0.3351,28069,5601,10078.5720
2,single|glidre@0.5,36266,469,3085,3593,0.1320,0.1155,0.1232,31373,1189,6019.0750
3,intersection|base@0.5;large@0.5,36266,1805,3348,2257,0.3503,0.4444,0.3918,31254,2564,13235.9120
4,union|base@0.5;large@0.5,36266,3029,13069,1033,0.1882,0.7457,0.3005,26089,7611,13235.9120
5,intersection|base@0.5;glidre@0.5,36266,368,641,3694,0.3647,0.0906,0.1451,32254,423,9176.4150
6,union|base@0.5;glidre@0.5,36266,2237,9518,1825,0.1903,0.5507,0.2829,28331,5341,9176.4150
7,intersection|large@0.5;glidre@0.5,36266,395,734,3667,0.3499,0.0972,0.1522,32208,420,16097.6470
8,union|large@0.5;glidre@0.5,36266,2772,11694,1290,0.1916,0.6824,0.2992,27462,6121,16097.6470
9,majority_2_of_3|base@0.5;large@0.5;glidre@0.5,36266,1902,3765,2160,0.3356,0.4682,0.3910,31052,2737,19254.9870


**reviewed_30_text**

,configuration,bags,TP,FP,FN,precision,recall,micro_f1,exact_match_bags,negative_bags_with_fp,total_member_inference_s
0,single|base@0.5,30,15,12,9,0.5556,0.6250,0.5882,13,3,3157.3400
1,single|large@0.5,30,23,15,1,0.6053,0.9583,0.7419,17,3,10078.5720
2,single|glidre@0.5,30,13,15,11,0.4643,0.5417,0.5000,16,0,6019.0750
3,intersection|base@0.5;large@0.5,30,15,3,9,0.8333,0.6250,0.7143,18,1,13235.9120
4,union|base@0.5;large@0.5,30,23,24,1,0.4894,0.9583,0.6479,11,5,13235.9120
5,intersection|base@0.5;glidre@0.5,30,8,4,16,0.6667,0.3333,0.4444,15,0,9176.4150
6,union|base@0.5;glidre@0.5,30,20,23,4,0.4651,0.8333,0.5970,14,3,9176.4150
7,intersection|large@0.5;glidre@0.5,30,12,3,12,0.8000,0.5000,0.6154,19,0,16097.6470
8,union|large@0.5;glidre@0.5,30,24,27,0,0.4706,1.0000,0.6400,14,3,16097.6470
9,majority_2_of_3|base@0.5;large@0.5;glidre@0.5,30,19,6,5,0.7600,0.7917,0.7755,19,1,19254.9870


,model,total_inference_s,mean_bag_wall_s,p95_bag_wall_s
0,base,3157.3400,0.0871,0.1450
1,large,10078.5720,0.2779,0.4610
2,glidre,6019.0750,0.1660,0.2938


,label_encoding_wall_s,load_wall_s,parameters,peak_worker_rss_bytes,pending_bags,started_utc
0,0.2387,15.6649,800241152,5831671808,36266,2026-10-06T15:08:10.743684+00:00
1,0.2033,19.3807,800241152,6953664512,17764,2026-10-06T17:51:57.947885+00:00


GLiDRE compatibility preflight seconds (excluded from inference cost): 15.011001332997694
Timing scope: Full-validation saved bag timings; GLiDRE includes previous failed attempts and session label encoding. Loading/preflight excluded. Ensemble cost sums all members.


## 7. Fair cutoff grid and quality–cost frontier

Each reference is examined separately. The leading observed individual and ensemble settings are displayed by micro-F1, with lower recorded cost breaking ties. A frontier configuration has no alternative with at least as high F1 and at most its cost, with one strict improvement. Equal-cost/equal-F1 ties are represented once.

The reviewed 30 are selected development cases; searching this grid does not turn them into an independent benchmark. The tables do not freeze a model, cutoff or fusion rule.

In [8]:
# Compare combinations with competitive individual cutoffs, using unchanged scores.
if comparison is not None:
    for reference in ('all_validation_distant_labels', 'reviewed_30_text'):
        frame = grid[grid['reference'] == reference].copy()
        ordered = frame.sort_values(['micro_f1', 'total_member_inference_s', 'configuration'],
                                    ascending=[False, True, True])
        display(Markdown('**' + reference + ': individual settings**'))
        show_table(ordered[ordered['members_count'] == 1][metric_columns].reset_index(drop=True))
        display(Markdown('**' + reference + ': ten leading ensemble settings in the fixed grid**'))
        show_table(ordered[ordered['members_count'] > 1][metric_columns].head(10).reset_index(drop=True))
        frontier = []
        for _, row in frame.iterrows():
            dominated = ((frame['micro_f1'] >= row['micro_f1'])
                         & (frame['total_member_inference_s'] <= row['total_member_inference_s'])
                         & ((frame['micro_f1'] > row['micro_f1'])
                            | (frame['total_member_inference_s'] < row['total_member_inference_s']))).any()
            if not dominated:
                frontier.append(row)
        frontier_frame = pd.DataFrame(frontier).sort_values(
            ['total_member_inference_s', 'micro_f1', 'configuration'], ascending=[True, False, True])
        frontier_frame = frontier_frame.drop_duplicates(['total_member_inference_s', 'micro_f1'])
        display(Markdown('**' + reference + ': observed F1/cost frontier**'))
        show_table(frontier_frame[metric_columns].reset_index(drop=True))


**all_validation_distant_labels: individual settings**

,configuration,bags,TP,FP,FN,precision,recall,micro_f1,exact_match_bags,negative_bags_with_fp,total_member_inference_s
0,single|large@0.9,36266,1524,1887,2538,0.4468,0.3752,0.4079,32289,1346,10078.5720
1,single|large@0.7,36266,2331,5367,1731,0.3028,0.5739,0.3964,30419,3363,10078.5720
2,single|large@0.5,36266,2698,9343,1364,0.2241,0.6642,0.3351,28069,5601,10078.5720
3,single|base@0.5,36266,2136,7074,1926,0.2319,0.5258,0.3219,28921,4873,3157.3400
4,single|base@0.7,36266,898,1881,3164,0.3231,0.2211,0.2625,31669,1466,3157.3400
5,single|glidre@0.5,36266,469,3085,3593,0.1320,0.1155,0.1232,31373,1189,6019.0750
6,single|glidre@0.7,36266,338,1655,3724,0.1696,0.0832,0.1116,31858,661,6019.0750
7,single|glidre@0.9,36266,154,591,3908,0.2067,0.0379,0.0641,32205,230,6019.0750
8,single|base@0.9,36266,62,119,4000,0.3425,0.0153,0.0292,32353,100,3157.3400


**all_validation_distant_labels: ten leading ensemble settings in the fixed grid**

,configuration,bags,TP,FP,FN,precision,recall,micro_f1,exact_match_bags,negative_bags_with_fp,total_member_inference_s
0,majority_2_of_3|base@0.5;large@0.7;glidre@0.7,36266,1732,2689,2330,0.3918,0.4264,0.4083,31743,2002,19254.9870
1,majority_2_of_3|base@0.5;large@0.7;glidre@0.5,36266,1766,2824,2296,0.3847,0.4348,0.4082,31686,2076,19254.9870
2,union|base@0.9;large@0.9,36266,1528,1920,2534,0.4432,0.3762,0.4069,32267,1370,13235.9120
3,intersection|base@0.5;large@0.7,36266,1649,2420,2413,0.4053,0.4060,0.4056,31864,1895,13235.9120
4,majority_2_of_3|base@0.5;large@0.7;glidre@0.9,36266,1676,2548,2386,0.3968,0.4126,0.4045,31796,1931,19254.9870
5,union|base@0.7;large@0.9,36266,1818,3165,2244,0.3648,0.4476,0.4020,31535,2264,13235.9120
6,union|base@0.9;large@0.7,36266,2333,5379,1729,0.3025,0.5743,0.3963,30415,3368,13235.9120
7,majority_2_of_3|base@0.5;large@0.5;glidre@0.7,36266,1874,3622,2188,0.3410,0.4613,0.3921,31113,2668,19254.9870
8,intersection|base@0.5;large@0.5,36266,1805,3348,2257,0.3503,0.4444,0.3918,31254,2564,13235.9120
9,majority_2_of_3|base@0.5;large@0.5;glidre@0.5,36266,1902,3765,2160,0.3356,0.4682,0.3910,31052,2737,19254.9870


**all_validation_distant_labels: observed F1/cost frontier**

...

,configuration,bags,TP,FP,FN,precision,recall,micro_f1,exact_match_bags,negative_bags_with_fp,total_member_inference_s
0,single|large@0.5,30,23,15,1,0.6053,0.9583,0.7419,17,3,10078.5720
1,single|large@0.7,30,20,10,4,0.6667,0.8333,0.7407,18,2,10078.5720
2,single|large@0.9,30,13,1,11,0.9286,0.5417,0.6842,19,0,10078.5720
3,single|base@0.5,30,15,12,9,0.5556,0.6250,0.5882,13,3,3157.3400
4,single|base@0.7,30,10,4,14,0.7143,0.4167,0.5263,16,0,3157.3400
5,single|glidre@0.5,30,13,15,11,0.4643,0.5417,0.5000,16,0,6019.0750
6,single|glidre@0.7,30,11,11,13,0.5000,0.4583,0.4783,15,0,6019.0750
7,single|glidre@0.9,30,6,6,18,0.5000,0.2500,0.3333,11,0,6019.0750
8,single|base@0.9,30,1,0,23,1.0000,0.0417,0.0800,11,0,3157.3400


**reviewed_30_text: ten leading ensemble settings in the fixed grid**

,configuration,bags,TP,FP,FN,precision,recall,micro_f1,exact_match_bags,negative_bags_with_fp,total_member_inference_s
0,majority_2_of_3|base@0.5;large@0.5;glidre@0.5,30,19,6,5,0.7600,0.7917,0.7755,19,1,19254.9870
1,majority_2_of_3|base@0.5;large@0.7;glidre@0.7,30,17,3,7,0.8500,0.7083,0.7727,20,0,19254.9870
2,majority_2_of_3|base@0.5;large@0.5;glidre@0.7,30,18,5,6,0.7826,0.7500,0.7660,19,1,19254.9870
3,majority_2_of_3|base@0.5;large@0.7;glidre@0.5,30,18,5,6,0.7826,0.7500,0.7660,20,0,19254.9870
4,majority_2_of_3|base@0.5;large@0.7;glidre@0.9,30,16,3,8,0.8421,0.6667,0.7442,19,0,19254.9870
5,union|base@0.9;large@0.5,30,23,15,1,0.6053,0.9583,0.7419,17,3,13235.9120
6,union|base@0.9;large@0.7,30,20,10,4,0.6667,0.8333,0.7407,18,2,13235.9120
7,majority_2_of_3|base@0.5;large@0.5;glidre@0.9,30,17,5,7,0.7727,0.7083,0.7391,18,1,19254.9870
8,union|large@0.7;glidre@0.9,30,23,16,1,0.5897,0.9583,0.7302,16,2,16097.6470
9,union|base@0.7;large@0.5,30,23,17,1,0.5750,0.9583,0.7188,16,3,13235.9120


**reviewed_30_text: observed F1/cost frontier**

,configuration,bags,TP,FP,FN,precision,recall,micro_f1,exact_match_bags,negative_bags_with_fp,total_member_inference_s
0,single|base@0.5,30,15,12,9,0.5556,0.6250,0.5882,13,3,3157.3400
1,union|base@0.5;glidre@0.9,30,17,15,7,0.5312,0.7083,0.6071,12,3,9176.4150
2,single|large@0.5,30,23,15,1,0.6053,0.9583,0.7419,17,3,10078.5720
3,majority_2_of_3|base@0.5;large@0.5;glidre@0.5,30,19,6,5,0.7600,0.7917,0.7755,19,1,19254.9870


## 8. Distant-label cohorts and per-relation results

The primary cutoff is broken down by distant-negative/positive bags and original bag size. Per-relation tables retain positive-label TP/FP/FN, precision, recall and F1. Differences against distant labels require text review before concluding that a prediction is semantically wrong.

In [9]:
# These tables reuse the same complete outputs; no additional model calls occur.
if comparison is not None:
    cohorts = pd.DataFrame(comparison['primary_cohorts'])
    for name in cohorts['cohort'].unique():
        display(Markdown('**' + name + '**'))
        show_table(cohorts[cohorts['cohort'] == name][metric_columns].reset_index(drop=True))
    relation_frame = pd.DataFrame(comparison['primary_per_relation'])
    display(Markdown('**Primary individual per-relation scores**'))
    show_table(relation_frame[relation_frame['configuration'].str.startswith('single|')][
        ['configuration', 'relation', 'TP', 'FP', 'FN', 'precision', 'recall', 'micro_f1']], max_rows=72)


**distant_NA**

,configuration,bags,TP,FP,FN,precision,recall,micro_f1,exact_match_bags,negative_bags_with_fp,total_member_inference_s
0,single|base@0.5,32392,0,5746,0,0.0000,0.0000,0.0000,27519,4873,3157.3400
1,single|large@0.5,32392,0,7089,0,0.0000,0.0000,0.0000,26791,5601,10078.5720
2,single|glidre@0.5,32392,0,1968,0,0.0000,0.0000,0.0000,31203,1189,6019.0750
3,intersection|base@0.5;large@0.5,32392,0,2792,0,0.0000,0.0000,0.0000,29828,2564,13235.9120
4,union|base@0.5;large@0.5,32392,0,10043,0,0.0000,0.0000,0.0000,24781,7611,13235.9120
5,intersection|base@0.5;glidre@0.5,32392,0,471,0,0.0000,0.0000,0.0000,31969,423,9176.4150
6,union|base@0.5;glidre@0.5,32392,0,7243,0,0.0000,0.0000,0.0000,27051,5341,9176.4150
7,intersection|large@0.5;glidre@0.5,32392,0,481,0,0.0000,0.0000,0.0000,31972,420,16097.6470
8,union|large@0.5;glidre@0.5,32392,0,8576,0,0.0000,0.0000,0.0000,26271,6121,16097.6470
9,majority_2_of_3|base@0.5;large@0.5;glidre@0.5,32392,0,3028,0,0.0000,0.0000,0.0000,29655,2737,19254.9870


**distant_positive**

,configuration,bags,TP,FP,FN,precision,recall,micro_f1,exact_match_bags,negative_bags_with_fp,total_member_inference_s
0,single|base@0.5,3874,2136,1328,1926,0.6166,0.5258,0.5676,1402,0,3157.3400
1,single|large@0.5,3874,2698,2254,1364,0.5448,0.6642,0.5986,1278,0,10078.5720
2,single|glidre@0.5,3874,469,1117,3593,0.2957,0.1155,0.1661,170,0,6019.0750
3,intersection|base@0.5;large@0.5,3874,1805,556,2257,0.7645,0.4444,0.5620,1426,0,13235.9120
4,union|base@0.5;large@0.5,3874,3029,3026,1033,0.5002,0.7457,0.5988,1308,0,13235.9120
5,intersection|base@0.5;glidre@0.5,3874,368,170,3694,0.6840,0.0906,0.1600,285,0,9176.4150
6,union|base@0.5;glidre@0.5,3874,2237,2275,1825,0.4958,0.5507,0.5218,1280,0,9176.4150
7,intersection|large@0.5;glidre@0.5,3874,395,253,3667,0.6096,0.0972,0.1677,236,0,16097.6470
8,union|large@0.5;glidre@0.5,3874,2772,3118,1290,0.4706,0.6824,0.5571,1191,0,16097.6470
9,majority_2_of_3|base@0.5;large@0.5;glidre@0.5,3874,1902,737,2160,0.7207,0.4682,0.5677,1397,0,19254.9870


**singleton**

...

,configuration,bags,TP,FP,FN,precision,recall,micro_f1,exact_match_bags,negative_bags_with_fp,total_member_inference_s
0,single|base@0.5,2974,841,1432,408,0.3700,0.6733,0.4776,1655,686,3157.3400
1,single|large@0.5,2974,1017,2181,232,0.3180,0.8143,0.4574,1431,791,10078.5720
2,single|glidre@0.5,2974,156,688,1093,0.1848,0.1249,0.1491,1773,166,6019.0750
3,intersection|base@0.5;large@0.5,2974,768,752,481,0.5053,0.6149,0.5547,1967,438,13235.9120
4,union|base@0.5;large@0.5,2974,1090,2861,159,0.2759,0.8727,0.4192,1221,983,13235.9120
5,intersection|base@0.5;glidre@0.5,2974,135,102,1114,0.5696,0.1081,0.1817,1939,44,9176.4150
6,union|base@0.5;glidre@0.5,2974,862,2018,387,0.2993,0.6902,0.4175,1557,741,9176.4150
7,intersection|large@0.5;glidre@0.5,2974,144,133,1105,0.5199,0.1153,0.1887,1919,49,16097.6470
8,union|large@0.5;glidre@0.5,2974,1029,2736,220,0.2733,0.8239,0.4105,1339,851,16097.6470
9,majority_2_of_3|base@0.5;large@0.5;glidre@0.5,2974,787,841,462,0.4834,0.6301,0.5471,1932,457,19254.9870


**size_above_20**

,configuration,bags,TP,FP,FN,precision,recall,micro_f1,exact_match_bags,negative_bags_with_fp,total_member_inference_s
0,single|base@0.5,75,63,116,9,0.3520,0.8750,0.5020,13,19,3157.3400
1,single|large@0.5,75,66,127,6,0.3420,0.9167,0.4981,12,20,10078.5720
2,single|glidre@0.5,75,20,58,52,0.2564,0.2778,0.2667,19,9,6019.0750
3,intersection|base@0.5;large@0.5,75,60,66,12,0.4762,0.8333,0.6061,26,16,13235.9120
4,union|base@0.5;large@0.5,75,69,177,3,0.2805,0.9583,0.4340,5,22,13235.9120
5,intersection|base@0.5;glidre@0.5,75,18,10,54,0.6429,0.2500,0.3600,33,2,9176.4150
6,union|base@0.5;glidre@0.5,75,65,164,7,0.2838,0.9028,0.4319,7,22,9176.4150
7,intersection|large@0.5;glidre@0.5,75,18,11,54,0.6207,0.2500,0.3564,33,1,16097.6470
8,union|large@0.5;glidre@0.5,75,68,174,4,0.2810,0.9444,0.4331,8,22,16097.6470
9,majority_2_of_3|base@0.5;large@0.5;glidre@0.5,75,62,73,10,0.4593,0.8611,0.5990,24,16,19254.9870


**Primary individual per-relation scores**

,configuration,relation,TP,FP,FN,precision,recall,micro_f1
0,single|base@0.5,/business/company/advisors,0,50,1,0.0000,0.0000,0.0000
1,single|base@0.5,/business/company/founders,30,99,7,0.2326,0.8108,0.3614
2,single|base@0.5,/business/company/majorshareholders,3,97,4,0.0300,0.4286,0.0561
3,single|base@0.5,/business/company/place_founded,1,20,35,0.0476,0.0278,0.0351
4,single|base@0.5,/business/location,2,281,19,0.0071,0.0952,0.0132
5,single|base@0.5,/business/person/company,254,678,37,0.2725,0.8729,0.4154
6,single|base@0.5,/film/film/featured_film_locations,0,25,1,0.0000,0.0000,0.0000
7,single|base@0.5,/location/administrative_division/country,94,368,31,0.2035,0.7520,0.3203
8,single|base@0.5,/location/country/administrative_divisions,38,281,78,0.1191,0.3276,0.1747
9,single|base@0.5,/location/country/capital,48,106,32,0.3117,0.6000,0.4103


## 9. Reviewed-30 errors and member contributions

This section uses the current confirmed annotations, all supplied sentences and saved sentence scores. The three individual cutoff settings and primary combinations are inspected. Pairwise contributions at 0.5 separate correct labels contributed only by one member from that member's exclusive false positives.

A weaker individual can still be complementary, but additional correct predictions, removed false positives, lost true positives and summed cost must be examined together. Contribution counts on this small reused sample do not establish population performance.
Detailed per-bag views require `ANALYZE_LOCAL_SCORES=True`; public report mode displays the aggregate diagnostic tables without local scores.


In [10]:
# Show per-case error counts and pairwise exclusive label contributions at 0.5.
if reviewed_bags:
    error_rows, contribution_rows = [], []
    for bag in reviewed_bags:
        for configuration, errors in bag['predictions'].items():
            error_rows.append({'sample': bag['sample_id'], 'pair': bag['head'] + ' → ' + bag['tail'],
                               'configuration': configuration, 'TP': len(errors['TP']),
                               'FP': len(errors['FP']), 'FN': len(errors['FN']),
                               'exact_match': not errors['FP'] and not errors['FN']})
        for contributions in bag['member_contributions_at_0_5']:
            contribution_rows.append({'sample': bag['sample_id'], **contributions})
    errors_frame = pd.DataFrame(error_rows)
    display(Markdown('**Reviewed error totals by configuration**'))
    show_table(errors_frame.groupby('configuration', sort=False)[['TP', 'FP', 'FN', 'exact_match']].sum().reset_index())
    contributions_frame = pd.DataFrame(contribution_rows)
    count_frame = contributions_frame.copy()
    for column in ('left_only_TP', 'left_only_FP', 'right_only_TP', 'right_only_FP'):
        count_frame[column] = count_frame[column].map(len)
    display(Markdown('**Exclusive pairwise contributions at 0.5**'))
    show_table(count_frame.groupby(['left', 'right'])[['left_only_TP', 'left_only_FP',
                                                     'right_only_TP', 'right_only_FP']].sum().reset_index())
else:
    print('Detailed reviewed errors require local cache analysis in section 5.')


Detailed reviewed errors require local cache analysis in section 5.


### 9.2 Complete texts, revised reference and error evidence

Each expandable case contains every selected sentence, current confirmed labels, separate distant labels, predictions and FP/FN lists. Bag maxima and sentence-level scores identify the evidence supporting or missing each label. Entity names and source text are escaped before HTML rendering. These views read inference outputs and never alter annotations.

In [11]:
# Render the reviewed cases as readable expandable evidence reports.
if reviewed_bags:
    import html
    def label_list(values):
        return ', '.join('<code>' + html.escape(v) + '</code>' for v in values) if values else '[]'
    panels = []
    for bag in reviewed_bags:
        content = [f"<p><strong>Confirmed:</strong> {label_list(bag['confirmed_relations'])}</p>",
                   f"<p><strong>Distant:</strong> {label_list(bag['distant_relations'])}</p>"]
        for i, sentence in enumerate(bag['sentences'], 1):
            content.append(f'<p><strong>Sentence {i}:</strong> {html.escape(sentence)}</p>')
        content.append('<table><tr><th>Configuration</th><th>Predicted relations</th><th>FP</th><th>FN</th></tr>')
        for configuration, outcome in bag['predictions'].items():
            content.append('<tr><td>' + html.escape(configuration) + '</td><td>'
                           + label_list(outcome['relations']) + '</td><td>' + label_list(outcome['FP'])
                           + '</td><td>' + label_list(outcome['FN']) + '</td></tr>')
        content.append('</table>')
        for model, values in bag['scores'].items():
            content.append('<details><summary>' + html.escape(model) + ': all bag and sentence scores</summary>')
            rows = [{'relation': label, 'bag_max': value,
                     **{f'sentence_{i+1}': evidence['relation_scores'][label]
                        for i, evidence in enumerate(values['evidence'])}}
                    for label, value in values['bag_maxima'].items()]
            content.append(pd.DataFrame(rows).sort_values('bag_max', ascending=False).to_html(
                index=False, escape=True, float_format=lambda v: f'{v:.4f}'))
            content.append('</details>')
        title = html.escape(bag['sample_id'] + ': ' + bag['head'] + ' → ' + bag['tail'])
        panels.append('<details class="review-case"><summary>' + title + '</summary>' + ''.join(content) + '</details>')
    display(HTML('<style>.review-case{margin:12px 0;padding:10px;border:1px solid #aaa;font-size:16px;}'
                 '.review-case summary{font-weight:600;cursor:pointer}.review-case table{width:100%;border-collapse:collapse}'
                 '.review-case th,.review-case td{padding:7px;border:1px solid #bbb;text-align:left;vertical-align:top;}'
                 '.review-case code{overflow-wrap:anywhere;white-space:normal;}</style>' + ''.join(panels)))


## 10. Annotation consistency and experiment status

The closing check rereads annotation files and rejects a reference changed during the run. Annotation edits after execution require rerunning the analysis cells; complete inference caches remain reusable. No final method selection or official-test evaluation is performed here.

In [12]:
# Verify the displayed report still uses the current annotation files.
reference_reader = importlib.reload(reference_reader)
_, final_reference_metadata = reference_reader.load_reviewed_reference(ROOT)
if comparison is not None:
    if comparison['reviewed_reference_metadata'] != final_reference_metadata:
        raise RuntimeError('Annotations changed during this run. Rerun section 5 onward; no inference is needed.')
    print('Comparison complete for:', ', '.join(comparison['models']))
    print('Reference hash verified:', final_reference_metadata['labels_sha256'])
    print('Original relex score caches were not overwritten.')
    print('No official test data or model evaluation was used.')


Comparison complete for: base, large, glidre
Reference hash verified: fb9972e04f461ce077ccde2c10d7811496fb711614be8d14344dde986fdb46ea
Original relex score caches were not overwritten.
No official test data or model evaluation was used.
